# Importing and preparing ImmoScout24 rental apartment data (buildings built before 1990)

**This Jupyter notebook does the following:**

- Imports web-scraped ImmoScout24 rental apartment data (Switzerland, buildings built in 1990 or earlier) from a CSV file
- Inspects the number of rows/columns and the data types
- Extracts the living area, gross rent, net rent and floor from raw strings using regular expressions (regex)
- Flags furnished, temporary and shared (WG) listings (they are kept, not deleted)
- Counts missing and duplicated values
- Removes apartments built in 1990, because Topic 1 covers buildings built **before** 1990
- Topic 1 extensions: cleans construction/renovation years, computes rent per m², extracts postcode and locality, and assigns renovation groups (not renovated / general renovation / energy renovation) using keywords
- Adds the boolean column `minergie` ("Minergie" in the listing card or in the description)
- Saves the prepared data to `Data/immoscout24_pre1990_prepared.csv`

The steps are the same as in `02_flatfox_data_preparation.ipynb`, so that both data sets can be combined later. The structure follows `Week_08/apartments_data_preparation_winterthur.ipynb`.

## Libraries and settings

In [1]:
# Libraries
import os
import re
import fnmatch
import numpy as np
import pandas as pd

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

# Get current working directory
print(os.getcwd())

c:\Users\DELL\Documents\applied_research_methods\Project_Topic1


## Importing data

In [2]:
# Show .csv - files in the folder 'Data'
flist = fnmatch.filter(os.listdir('./Data'), '*.csv')
for i in flist:
    print(i)

# Read the data to a pandas data frame
# (descriptions contain line breaks inside quotes; read_csv handles this)
df = pd.read_csv('./Data/immoscout24_mieten_400.csv', sep=',', encoding='utf-8')

# Rename columns with spaces or umlauts, so that they are easier to use
df = df.rename(columns={'Baujahr': 'year_built',
                        'renovationed year': 'year_renovated',
                        'Wohnfläche': 'living_area',
                        'etage': 'floor_raw'})

# Show first records of data frame
df.head()

AMTOVZ_CSV_WGS84.csv
apartments_pre1990_clean.csv
apartments_pre1990_combined.csv
apartments_pre1990_enriched.csv
apartments_pre1990_renovation_details.csv
flatfox_api_attributes.csv
flatfox_pre1991_prepared.csv
flatfox_switzerland_rent_apartments_pre1991.csv
immoscout24_mieten_400.csv
immoscout24_pre1990_prepared.csv


,web_scraper_order,web_scraper_start_url,listing,listing-href,title,address,rooms,rent_gross,rent_net,description,floor_raw,year_built,living_area,year_renovated
0,1795642997-1,https://www.immoscout24.ch/de/wohnung/mieten/l...,GoldPartner1 / 7GoldPartnerBalkon / TerrasseHa...,https://www.immoscout24.ch/mieten/4003651317,"""4.5-Zimmer-Wohnung mit kostenloser Kaution""","Croix Fédérale 44, 2300 La Chaux-de-Fonds",4.5,CHF 1'485.–,CHF 1'200.–,Die Wohnung besteht aus:\n- Vollständig ausges...,3,1972,83 m2,2012.0
1,1795643006-2,https://www.immoscout24.ch/de/wohnung/mieten/l...,GoldPartner1 / 6GoldPartnerBalkon / TerrasseHa...,https://www.immoscout24.ch/mieten/4003652268,"""Willkommen in Luterbach - Kinderfreundliche Ü...","Grüttbachstr. 6, 4542 Luterbach",4.5,CHF 1'990.–,CHF 1'630.–,Wir vermieten per sofort oder nach Vereinbarun...,1,1988,108 m2,NaN
2,1795643013-3,https://www.immoscout24.ch/de/wohnung/mieten/l...,GoldPartner1 / 8GoldPartnerHaustiere erlaubt4....,https://www.immoscout24.ch/mieten/4003652272,"""Platz für die ganze Familie!""","Neuhofstr. 20, 5600 Lenzburg",4.5,CHF 1'760.–,CHF 1'470.–,Wir vermieten per sofort oder nach Vereinbarun...,EG,1960,80 m2,NaN
3,1795643024-4,https://www.immoscout24.ch/de/wohnung/mieten/l...,GoldPartner1 / 6GoldPartnerBalkon / TerrasseHa...,https://www.immoscout24.ch/mieten/4003653122,"""Bezahlbarer Platz für die ganze Familie.""","Aathalstr. 19, 8610 Uster",5.5,CHF 2'220.–,CHF 1'860.–,Diese 5.5 Zimmer Wohnung zeichnet sich durch f...,2,1982,116 m2,NaN
4,1795643036-5,https://www.immoscout24.ch/de/wohnung/mieten/l...,GoldPartner1 / 7GoldPartnerBalkon / TerrasseLi...,https://www.immoscout24.ch/mieten/4003653124,"""Schöne, helle Wohnung mit Balkon""","Rte de Bertigny 2/4, 1700 Fribourg",2.0,CHF 1'275.–,CHF 1'130.–,Lassen Sie sich von dieser lichtdurchfluteten ...,2,1959,52 m2,1996.0


## Count number of rows and columns in the data frame

In [3]:
# Dimension (rows, columns)
print('Dimension:', df.shape)

# Number of rows
print('Number of rows:', df.shape[0])

# Number of columns
print('Number of columns:', df.shape[1])

Dimension: (398, 14)
Number of rows: 398
Number of columns: 14


## Get data types (raw-format from web scraping)

In [4]:
# Get data types (since pandas 3.0, text columns have the data type 'str';
# in older versions of pandas, they have the data type 'object')
df.dtypes

web_scraper_order            str
web_scraper_start_url        str
listing                      str
listing-href                 str
title                        str
address                      str
rooms                    float64
rent_gross                   str
rent_net                     str
description                  str
floor_raw                    str
year_built                 int64
living_area                  str
year_renovated           float64
dtype: object

## Extract and save relevant information from raw data using regular expressions (regex)

### Number of rooms

On ImmoScout24, the column `rooms` already contains numbers like `3.5` (data type `float64`), so no regex is needed. Only the data type is checked.

In [5]:
# Make sure 'rooms' is numeric (non-numeric values become missing)
df['rooms'] = pd.to_numeric(df['rooms'], errors='coerce')

print(df['rooms'].value_counts().sort_index())

rooms
1.0    12
1.5     7
2.0    16
2.5    48
3.0    38
3.5    96
4.0    34
4.5    93
5.0    12
5.5    31
6.0     4
6.5     3
7.0     1
7.5     1
8.0     2
Name: count, dtype: int64


### Extract living area

The column `living_area` contains values like `"83 m2"`. If it is empty, the area is taken from the listing card (e.g. `"4.5 Zimmer, 83m², CHF 1'485.–"`).

In [6]:
# Extract values from 'living_area' strings, e.g. '83 m2'
area = []
for a, l in zip(df['living_area'].fillna('').astype(str), df['listing'].fillna('').astype(str)):
    d1 = re.findall(r'(\d+)\s*m2', re.sub(r"['’]", '', a))     # remove thousands separators ' and ’
    if not d1:
        d1 = re.findall(r'(\d+)\s*m²', re.sub(r"['’]", '', l))  # fallback: listing card
    try:
        d2 = int(d1[0])
    except:
        d2 = None
    area.append(d2)

# Save as new variable in the pandas data frame
df['area'] = pd.Series(area, dtype="Int64")

# Print first 5 values
print(df['living_area'].head(5), '\n')
print(df['area'].head(5))

0     83 m2
1    108 m2
2     80 m2
3    116 m2
4     52 m2
Name: living_area, dtype: str 

0     83
1    108
2     80
3    116
4     52
Name: area, dtype: Int64


### Extract gross rent and net rent

The columns `rent_gross` and `rent_net` contain values like `"CHF 1'485.–"`. The thousands separator `'` (and the typographic `’`) is removed, otherwise `"1'485"` would be read as `1`. `"auf Anfrage"` (price on request) becomes missing.

- `price` = gross rent (including Nebenkosten), like `price` in the flatfox data
- `price_net` = net rent (without Nebenkosten); about a quarter of the listings has no net rent

In [7]:
# Function: first number in a rent string, e.g. "CHF 1'485.–" -> 1485
def extract_rent(series):
    values = []
    for i in series.fillna('').astype(str):
        d1 = re.findall('[0-9]+', re.sub(r"[\s'’]", '', i))  # remove spaces and thousands separators ' and ’
        try:
            d2 = int(d1[0])
        except:
            d2 = None
        values.append(d2)
    return pd.Series(values, dtype="Int64")

# Save as new variables in the pandas data frame
df['price'] = extract_rent(df['rent_gross'])
df['price_net'] = extract_rent(df['rent_net'])

# Print first 5 values
print(df[['rent_gross', 'price', 'rent_net', 'price_net']].head(5), '\n')
print('Price on request (gross rent missing):', df['price'].isna().sum())

    rent_gross  price     rent_net  price_net
0  CHF 1'485.–   1485  CHF 1'200.–       1200
1  CHF 1'990.–   1990  CHF 1'630.–       1630
2  CHF 1'760.–   1760  CHF 1'470.–       1470
3  CHF 2'220.–   2220  CHF 1'860.–       1860
4  CHF 1'275.–   1275  CHF 1'130.–       1130 

Price on request (gross rent missing): 6


### Extract floor

The column `floor_raw` contains values like `"3"`, `"EG"` (ground floor) or `"-1"` (basement). `"EG"` is converted to `0`.

In [8]:
df['floor'] = pd.to_numeric(df['floor_raw'].replace('EG', '0'), errors='coerce').astype('Int64')

print(df['floor'].value_counts(dropna=False).sort_index())

floor
-3        1
-1        2
0        57
1        92
2       108
3        67
4        24
5        21
6        12
7         4
8         1
9         1
16        1
<NA>      7
Name: count, dtype: Int64


### Flag furnished, temporary and shared listings

ImmoScout24 has no separate columns for these, so they are found with keywords in the title and description. These rows are **kept** and only flagged, because they can bias the rent comparison.

- `furnished`: "möbliert", "meublé" or "arredato" (but not "unmöbliert" / "nicht möbliert")
- `temporary`: "befristet" (temporary lease)
- `shared`: a room in a shared flat ("WG-Zimmer")

In [9]:
text = (df['title'].fillna('') + ' ' + df['description'].fillna('')).str.lower()

# Remove negative phrases first, so that 'unmöbliert' does not count as 'möbliert'
text_furn = text.str.replace(r'unmöbliert|nicht möbliert|non meublé|non arredato', '', regex=True)

df['furnished'] = text_furn.str.contains(r'möbliert|meublé|arredato', regex=True)
df['temporary'] = text.str.contains('befristet')
df['shared'] = text.str.contains(r'wg-zimmer|wg zimmer', regex=True)

print('Furnished:', df['furnished'].sum(), '| Temporary:', df['temporary'].sum(), '| Shared (WG):', df['shared'].sum(), '\n')

# Show the flagged listings
df.loc[df['furnished'] | df['temporary'] | df['shared'], ['address', 'title', 'price', 'furnished', 'temporary', 'shared']]

Furnished: 25 | Temporary: 5 | Shared (WG): 2 



,address,title,price,furnished,temporary,shared
48,"av. de Champel 59, 1206 Genève","""Champel, wunderschöne 6-Zimmer-Wohnung""",4929,False,True,False
87,"Via Canova 7, 6900 Lugano","""3.5-Zimmer-Wohnung im Zentrum von Lugano zu v...",6400,True,False,False
93,"av. de Champel 63, 1206 Genève","""Champel: 4-Zimmer-Wohnung mit Ausrichtung auf...",3648,False,True,False
99,"Av. du Mont - d'Or 55, 1007 Lausanne","""Wunderschöne Wohnung mit Charme / Ein Monat m...",2045,False,True,False
111,"Rütistrasse 29, 8702 Zollikon","""Zürichsee im Blick und viel Raum für das Fami...",4890,True,False,False
126,"Rue de Bernex 308, 1233 Bernex","""Charmanter möbliertes Apartment mit Balkon""",2300,True,False,False
136,"Rue Louis - Curval 10, 1206 Genève","""Champel: Wunderschöne Wohnung mit 3 Schlafzim...",4120,False,True,False
153,"Route de Glion 16, 1820 Territet","""Territet · 3,5 Zimmer renoviert, Seeblick und...",2800,True,False,False
154,"Eichmattstrasse 10, 6005 Luzern","""Charmante 2.5-Zimmer-Wohnung teilmöbliert mit...",1985,True,False,False
155,"Eichmattstrasse 10, 6005 Luzern","""Charmante 1.5-Zimmer-Wohnung mit Balkon an ru...",1580,True,False,False


### Get data types of all variables including the new ones

In [10]:
df.dtypes

web_scraper_order            str
web_scraper_start_url        str
listing                      str
listing-href                 str
title                        str
address                      str
rooms                    float64
rent_gross                   str
rent_net                     str
description                  str
floor_raw                    str
year_built                 int64
living_area                  str
year_renovated           float64
area                       Int64
price                      Int64
price_net                  Int64
floor                      Int64
furnished                   bool
temporary                   bool
shared                      bool
dtype: object

## Count and identify missing values (if any)

In [11]:
# Count missing values
print(pd.isna(df).sum())

# Identify rows with missing values
df[df.isna().any(axis=1)].head()

web_scraper_order          0
web_scraper_start_url      0
listing                    0
listing-href               0
title                      0
address                    0
rooms                      0
rent_gross                 0
rent_net                 110
description                1
floor_raw                  7
year_built                 0
living_area               11
year_renovated           105
area                       9
price                      6
price_net                110
floor                      7
furnished                  0
temporary                  0
shared                     0
dtype: int64


,web_scraper_order,web_scraper_start_url,listing,listing-href,title,address,rooms,rent_gross,rent_net,description,...,year_built,living_area,year_renovated,area,price,price_net,floor,furnished,temporary,shared
1,1795643006-2,https://www.immoscout24.ch/de/wohnung/mieten/l...,GoldPartner1 / 6GoldPartnerBalkon / TerrasseHa...,https://www.immoscout24.ch/mieten/4003652268,"""Willkommen in Luterbach - Kinderfreundliche Ü...","Grüttbachstr. 6, 4542 Luterbach",4.5,CHF 1'990.–,CHF 1'630.–,Wir vermieten per sofort oder nach Vereinbarun...,...,1988,108 m2,NaN,108,1990,1630,1,False,False,False
2,1795643013-3,https://www.immoscout24.ch/de/wohnung/mieten/l...,GoldPartner1 / 8GoldPartnerHaustiere erlaubt4....,https://www.immoscout24.ch/mieten/4003652272,"""Platz für die ganze Familie!""","Neuhofstr. 20, 5600 Lenzburg",4.5,CHF 1'760.–,CHF 1'470.–,Wir vermieten per sofort oder nach Vereinbarun...,...,1960,80 m2,NaN,80,1760,1470,0,False,False,False
3,1795643024-4,https://www.immoscout24.ch/de/wohnung/mieten/l...,GoldPartner1 / 6GoldPartnerBalkon / TerrasseHa...,https://www.immoscout24.ch/mieten/4003653122,"""Bezahlbarer Platz für die ganze Familie.""","Aathalstr. 19, 8610 Uster",5.5,CHF 2'220.–,CHF 1'860.–,Diese 5.5 Zimmer Wohnung zeichnet sich durch f...,...,1982,116 m2,NaN,116,2220,1860,2,False,False,False
10,1795643104-11,https://www.immoscout24.ch/de/wohnung/mieten/l...,"GoldPartnerNeu1 / 13GoldPartner3.5 Zimmer, 74m...",https://www.immoscout24.ch/mieten/4003655824,"""Erstbezug nach Sanierung moderne 3.5-Zimmer-...","Holzgasse 27, 4537 Wiedlisbach",3.5,CHF 1'650.–,CHF 1'500.–,Per sofort oder nach Vereinbarung vermieten wi...,...,1962,74 m2,NaN,74,1650,1500,<NA>,False,False,False
13,1795643137-14,https://www.immoscout24.ch/de/wohnung/mieten/l...,"GoldPartner1 / 10GoldPartnerLift3.5 Zimmer, 79...",https://www.immoscout24.ch/mieten/4002168991,"""Willkommensgeschenk - Wohnen direkt an der Re...","Seidenhofstr. 1, 6003 Luzern",3.5,CHF 2'890.–,CHF 2'660.–,Wir vermieten nach Vereinbarung diese attrakti...,...,1940,79 m2,NaN,79,2890,2660,2,False,False,False


## Count and identify duplicated values (if any)

- `listing-href` identifies a listing. Rows with the same URL were scraped twice and are removed.
- Same address, rooms and rent can also be **different** flats in the same building (e.g. two flats on different floors), so these are only shown, not removed.

In [12]:
# Exact duplicates: same listing URL scraped more than once
print('Duplicated listing URLs:', df.duplicated(subset=['listing-href']).sum())

# Possible duplicates: same address, rooms and rent
dup_cols = ['address', 'rooms', 'rent_gross']
print('Same address, rooms and rent:', df.duplicated(subset=dup_cols).sum())

# Identify rows with possible duplicates (all occurrences)
df[df.duplicated(subset=dup_cols, keep=False)].sort_values('address')[dup_cols + ['floor_raw', 'listing-href']]

Duplicated listing URLs: 0
Same address, rooms and rent: 3


,address,rooms,rent_gross,floor_raw,listing-href
41,"Ch. de Monséjour 19, 1700 Fribourg",3.5,CHF 1'705.–,2,https://www.immoscout24.ch/mieten/4003534191
355,"Ch. de Monséjour 19, 1700 Fribourg",3.5,CHF 1'705.–,4,https://www.immoscout24.ch/mieten/4003147291
44,"Ch. des Sources 16, 2013 Colombier NE",3.5,CHF 2'100.–,2,https://www.immoscout24.ch/mieten/4003541713
56,"Ch. des Sources 16, 2013 Colombier NE",3.5,CHF 2'100.–,1,https://www.immoscout24.ch/mieten/4003610396
138,"Kirchbergstrasse 91, 8207 Schaffhausen",5.5,CHF 2'790.–,NaN,https://www.immoscout24.ch/mieten/4003627142
139,"Kirchbergstrasse 91, 8207 Schaffhausen",5.5,CHF 2'790.–,NaN,https://www.immoscout24.ch/mieten/4003627143


In [13]:
# Remove exact duplicates (same listing URL)
df = df.drop_duplicates(subset=['listing-href']).reset_index(drop=True)
print('Dimension after removing duplicated URLs:', df.shape)

Dimension after removing duplicated URLs: (398, 21)


## Topic 1: additional preparation

The following steps are not part of the Week 8 example. They are needed for the research question of Topic 1 (green premium of energy-renovated older apartments in Switzerland).

### Clean construction year and renovation year

- `year_built`: values below 1000 are set to missing. Only years up to 1989 are kept.
- Apartments built in 1990 are **removed**: the scrape filter (`bg=1990`) includes 1990, but Topic 1 covers buildings built **before** 1990.
- `year_renovated`: years in the future (after 2026) and years not after the construction year are set to missing.

In [14]:
# Convert to numbers (non-numeric values become missing)
df['year_built'] = pd.to_numeric(df['year_built'], errors='coerce')
df['year_renovated'] = pd.to_numeric(df['year_renovated'], errors='coerce')

# Clean construction year
df['year_built_clean'] = df['year_built'].where(df['year_built'].between(1000, 1989))

# Remove apartments built in 1990 or later (Topic 1: buildings built before 1990)
built_1990 = df['year_built'] >= 1990
print('Apartments built in 1990 or later (removed):', built_1990.sum())
df = df.loc[~built_1990].reset_index(drop=True)
print('Dimension after removal:', df.shape, '\n')

# Clean renovation year
valid_reno = (df['year_renovated'].between(1900, 2026)
              & (df['year_renovated'] > df['year_built_clean'].fillna(0)))
df['year_renovated_clean'] = df['year_renovated'].where(valid_reno)

# Show the values that were set to missing
print('year_built set to missing:')
print(df.loc[df['year_built'].notna() & df['year_built_clean'].isna(), ['address', 'year_built']], '\n')
print('year_renovated set to missing:')
print(df.loc[df['year_renovated'].notna() & df['year_renovated_clean'].isna(),
             ['address', 'year_built', 'year_renovated']])

Apartments built in 1990 or later (removed): 11
Dimension after removal: (387, 22) 

year_built set to missing:
Empty DataFrame
Columns: [address, year_built]
Index: [] 

year_renovated set to missing:
                               address  year_built  year_renovated
10          Amselstr. 24, 4104 Oberwil        1987          2027.0
312  Arlesheimerstr. 24, 4147 Aesch BL        1982          1982.0
373    Grabenwiese 8, 8484 Weisslingen        1960          2027.0


### Plausibility check of living area and rent per m²

- Living areas below 10 m² or above 500 m² are set to missing.
- `rent_per_m2` = gross rent / living area (same definition as in the flatfox data).
- `rent_net_per_m2` = net rent / living area (only where the net rent is known).
- Shared rooms (WG) are not apartments; their rent per m² is left empty.

In [15]:
# Plausible living area
df['area_clean'] = df['area'].where(df['area'].between(10, 500))
print('Area set to missing:')
print(df.loc[df['area'].notna() & df['area_clean'].isna(), ['address', 'living_area']], '\n')

# Rent per m² (not for shared rooms)
apartment = ~df['shared']
df['rent_per_m2'] = (df['price'] / df['area_clean']).where(apartment).round(2)
df['rent_net_per_m2'] = (df['price_net'] / df['area_clean']).where(apartment).round(2)

print(df[['rent_per_m2', 'rent_net_per_m2']].describe())

Area set to missing:
Empty DataFrame
Columns: [address, living_area]
Index: [] 

       rent_per_m2  rent_net_per_m2
count        371.0            275.0
mean     25.095364        21.826655
std       9.597842         8.227004
min           9.76              9.6
25%         18.935            16.15
50%          22.35            19.41
75%          28.07           24.745
max           70.0            57.69


### Extract postcode and locality

The postcode and locality are extracted from `address` (e.g. `'Croix Fédérale 44, 2300 La Chaux-de-Fonds'` or `'4455 Zunzgen'`). They are needed later to merge municipality-level data (e.g. SFSO "Regionalporträts") as control variables for location.

In [16]:
# Postcode and locality: the last '1234 Name' part of the address
df['zip'] = pd.to_numeric(df['address'].str.extract(r'(\d{4})\s+[^,\d][^,]*$', expand=False), errors='coerce').astype('Int64')
df['locality'] = df['address'].str.extract(r'\d{4}\s+([^,\d][^,]*)$', expand=False).str.strip()

print('Listings without postcode:', df['zip'].isna().sum())
print('Number of different localities:', df['locality'].nunique(), '\n')

# Most frequent localities
print(df['locality'].value_counts().head(10))

Listings without postcode: 0
Number of different localities: 204 

locality
Basel          34
Genève         16
St. Gallen     13
Fribourg       12
Zürich         10
Bern            8
Biel/Bienne     7
Luzern          6
Binningen       6
Bellinzona      5
Name: count, dtype: int64


### Renovation groups (keyword search in the description)

Each listing is assigned to one group, with the same keyword lists as in the flatfox notebook:

1. **energy renovation**: the description contains an energy keyword (e.g. Wärmepumpe, Minergie, new windows, insulation)
2. **general renovation**: a general renovation keyword (e.g. renoviert, neue Küche) **or** a valid renovation year
3. **not renovated**: none of the above

Keywords are matched as lower-case substrings. This is a simple heuristic: listings may be assigned to the wrong group (see limitations of Topic 1). Review a sample of each group and adjust the keyword lists if needed.

In [17]:
energy_keywords = [
    # German
    'energetisch', 'minergie', 'wärmepumpe', 'fassade', 'gebäudehülle', 'dämmung', 'gedämmt',
    'neue fenster', 'fenster ersetzt', 'fenstersanierung', '3-fach', '3fach', 'dreifach',
    'photovoltaik', 'solaranlage', 'pellet', 'neue heizung', 'heizung ersetzt',
    # French
    'pompe à chaleur', 'isolation thermique', 'façade', 'triple vitrage', 'photovoltaïque',
    'panneaux solaires', 'haute performance énergétique',
    # Italian
    'pompa di calore', 'facciate isolate', 'cappotto', 'tripli vetri', 'triplo vetro', 'fotovoltaico',
]

general_keywords = [
    # German
    'renoviert', 'renovation', 'saniert', 'sanierung', 'erneuert', 'modernisiert', 'umgebaut',
    'neue küche', 'neues bad', 'neue bad',
    # French
    'rénové', 'rénovation', 'refait', 'cuisine neuve',
    # Italian
    'ristrutturat', 'rinnovat', 'risanament',
]

desc = df['description'].fillna('').str.lower()
df['keyword_energy'] = desc.apply(lambda t: any(k in t for k in energy_keywords))
df['keyword_general'] = desc.apply(lambda t: any(k in t for k in general_keywords))

df['renovation_group'] = np.select(
    [df['keyword_energy'],
     df['keyword_general'] | df['year_renovated_clean'].notna()],
    ['energy renovation', 'general renovation'],
    default='not renovated')

print(df['renovation_group'].value_counts(), '\n')

# Mean gross rent per m² by renovation group (first descriptive comparison, no controls)
df.groupby('renovation_group')['rent_per_m2'].agg(['count', 'mean', 'median']).round(2)

renovation_group
general renovation    275
not renovated          81
energy renovation      31
Name: count, dtype: int64 



,count,mean,median
renovation_group,,,
energy renovation,31,24.59,23.93
general renovation,260,25.04,22.09
not renovated,80,25.46,23.07


In [18]:
# Review: listings in the energy renovation group and the keywords found
def found_keywords(text, keywords):
    return ', '.join(k for k in keywords if k in text)

review = df[df['renovation_group'] == 'energy renovation'][['address', 'year_renovated_clean']].copy()
review['energy_keywords_found'] = desc[review.index].apply(lambda t: found_keywords(t, energy_keywords))
review

,address,year_renovated_clean,energy_keywords_found
2,"Neuhofstr. 20, 5600 Lenzburg",NaN,neue fenster
9,"Holzgasse 27, 4537 Wiedlisbach",NaN,wärmepumpe
43,"Ch. des Sources 16, 2013 Colombier NE",2024.0,photovoltaik
52,"Via Franco Zorzi 25a, 6500 Bellinzona",NaN,photovoltaik
55,"Ch. des Sources 16, 2013 Colombier NE",2024.0,photovoltaik
58,"Hohlstrasse 90, 8004 Zürich",2024.0,minergie
72,"Ch. des Sources 16a, 2013 Colombier NE",2024.0,photovoltaik
110,"Rütistrasse 29, 8702 Zollikon",NaN,dreifach
167,"Rickenstrasse 12, 8732 Neuhaus",2026.0,"wärmepumpe, neue heizung"
171,"Rathausgasse 24, 3011 Bern",1989.0,"neue fenster, 3-fach"


### Minergie (boolean)

The column `minergie` shows whether a listing is Minergie-certified. It combines two sources:

1. the feature "Minergie zertifiziert" on the ImmoScout24 listing card (column `listing`)
2. the word "Minergie" in the description

Unlike flatfox, ImmoScout24 has no public Web API, so there is no third source. `False` means: no Minergie mentioned (the building can still be Minergie-certified).

In [19]:
# Source 1: feature on the listing card
minergie_card = df['listing'].fillna('').str.lower().str.contains('minergie')

# Source 2: 'minergie' in the description
minergie_description = df['description'].fillna('').str.lower().str.contains('minergie')

df['minergie'] = (minergie_card | minergie_description).astype('boolean')

print('minergie:')
print(df['minergie'].value_counts(dropna=False), '\n')
print('Source: listing card', minergie_card.sum(), '| description', minergie_description.sum(),
      '| both', (minergie_card & minergie_description).sum(), '\n')

# Minergie by renovation group
pd.crosstab(df['renovation_group'], df['minergie'].astype('str'), margins=True)

minergie:
minergie
False    380
True       7
Name: count, dtype: Int64 

Source: listing card 7 | description 3 | both 3 



minergie,False,True,All
renovation_group,,,
energy renovation,25,6,31
general renovation,274,1,275
not renovated,81,0,81
All,380,7,387


### Save data to file

In [20]:
df.to_csv('./Data/immoscout24_pre1990_prepared.csv',
          sep=",",
          encoding='utf-8',
          index=False)

print('Saved:', df.shape[0], 'listings')

Saved:

 387 listings


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [21]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
NT
Windows | 10
Datetime: 2026-11-26 15:44:57
Python Version: 3.11.8
-----------------------------------
